# Perfos visualization

* [Setup](#Setup)
* [Global statistics](#global-stats)
  * [1°) Stats AREA](#stats-area)
  * [2°) Stats WSE](#stats-wse)
* [Plots of spatial and source repartitions](#repartition)
* [Plots of errors and distributions](#plots-errors)
    * [Box plots of AREA error as a function of altitude](#box-plots-area-altitude)
    * [Characteristics of the measured LakeSP_Prior fractional area error without any filtering and severe filtering and lakes distributions](#resume_area)
    * [Box plots of WSE error as a function of altitude](#box-plots-wse-altitude)
    * [Characteristics of the measured LakeSP_Prior WSE error without any filtering and severe filtering and lakes distributions](#resume_wse)

In [ ]:
import os
import sys
from datetime import datetime, date
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely import wkt
import geopandas as gpd
import requests
from shapely.geometry import shape
import json
from matplotlib.ticker import LogLocator, FuncFormatter

## Setup

##### Environnement and paths

In [ ]:
# os.environ["SWOT_HYDROLOGY_TOOLBOX"] = "path/to/swot-hydrology-toolbox"
os.environ["SWOT_HYDROLOGY_TOOLBOX"] = os.popen("git rev-parse --show-toplevel").read().strip()
sht_path = os.environ["SWOT_HYDROLOGY_TOOLBOX"]
print(sht_path)

In [ ]:
input_dir = os.path.join(sht_path, "performances/examples/input")
output_dir = os.path.join(sht_path, "performances/examples/output")

##### Path to the directory containing the .png images that were created

In [ ]:
today = date.today()
cur_date = today.strftime("%Y%m%d")
# cur_date = '20260921'
cur_date

In [ ]:
start_calval = datetime.strptime("28.03.2023 22:02:06", "%d.%m.%Y %H:%M:%S")
end_calval = datetime.strptime("11.07.2023 22:02:06", "%d.%m.%Y %H:%M:%S") 
start_science = datetime.strptime("26.07.2023 11:37:56", "%d.%m.%Y %H:%M:%S")
end_date = datetime.now() 

In [ ]:
out_dir = os.path.join(output_dir, "lakes_perfos_visu_{}".format(cur_date))
if not os.path.exists(out_dir):
    os.makedirs(out_dir)
os.path.relpath(out_dir, start=sht_path)

In [ ]:
# Change the names of the sources so that they match those of the tables in the publication
list_sources = {
 'bafu':'BAFU',
 'nve':'NVE',
 'smhi':'SMHI',
 'usgs':'USGS',
 'lake_us':'US Tier-1',
 'eccc_gnss':'ECCC',
 'cehq':'MELCCFP',
 'hydroquebec':'HQ',
 'udes':'Université de Sherbrooke',
 'spence':'Chris Spence',
 'sertit':'SERTIT',
 'oecs_lake':'SERTIT',
 'locss_pyrenees':'LOCSS',
 'ceara_wse':'FUNCEME',
 'curuai':'SGB',
 'sget':'DGPRE, AMMA-CATCH, UJKZ',
 'tibet':'ITPR',
 'PleiadesXS_TP':'Pléiades MS',
 'SLUM_TP':'Pléiades PMS',
 'RCM_TP':'RCM 5MCP',
}

# Dictionnary with colors and markers for each sources
style_dict = {
    "BAFU": {"color": "#6CBF6C", "marker": "o"},  
    "NVE": {"color": "#E8A0C9", "marker": "s"},   
    "SMHI": {"color": "#F2E86D", "marker": "^"}, 
    "USGS": {"color": "#F59B4B", "marker": "o"}, 
    "US Tier-1": {"color": "#AA72B8", "marker": "s"}, 
    "ECCC": {"color": "#6CBF6C", "marker": "^"}, 
    "MELCCFP": {"color": "#E56A6A", "marker": "o"},  
    "HQ": {"color": "#F2E86D", "marker": "s"},     
    "Université de Sherbrooke": {"color": "#5E95C8", "marker": "^"},
    "Chris Spence": {"color": "#AA72B8", "marker": "o"},       
    "SERTIT": {"color": "#E56A6A", "marker": "s"}, 
    "LOCSS": {"color": "#F59B4B", "marker": "^"},   
    "FUNCEME": {"color": "#5E95C8", "marker": "o"}, 
    "SGB": {"color": "#6CBF6C", "marker": "s"},     
    "DGPRE, AMMA-CATCH, UJKZ": {"color": "#E56A6A", "marker": "^"},
    "ITPR": {"color": "#B88A5A", "marker": "o"},
    
    "Pléiades PMS": {"color": "#E56A6A", "marker": "^"},
    "Pléiades MS": {"color": "#5E95C8", "marker": "o"},
    "RCM 5MCP": {"color": "#F2E86D", "marker": "s"},
}

list_sources_area = ['Pléiades MS','Pléiades PMS', 'RCM 5MCP']
style_dict_wse = {k: v for k, v in style_dict.items() if k not in list_sources_area}

##### Functions

In [ ]:
# [ALL] Estimate percentiles
def _p50_p68(x):
    x = np.asarray(x, dtype=float)
    print("50%ile {:.1f}".format(np.nanpercentile(x, 50)))
    print("|68%ile| {:.1f}".format(np.nanpercentile(np.abs(x), 68.27)))
    return np.nanpercentile(x, 50), np.nanpercentile(np.abs(x), 68.27)

In [ ]:
# [ALL] Extract number of matchups
def _nb_matchups(x):
    print("Number of matchups: {}".format(len(x)))
    return len(x)

In [ ]:
# [ALL] Extract number of PLD lakes
def _nb_pld(x):
    print("Number of PLD lakes: {}".format(x['_fuid'].str.split(':').str[1].nunique()))
    return x['_fuid'].str.split(':').str[1].nunique()

In [ ]:
# [AREA] Lake size between > 100² et <= 250² m²
cat_100x100_to_250x250_m2=f'> 100² and <= 250² m²'
def _filtered_100x100_to_250x250_m2(x):
    return x[(x["p_ref_area"] > 0.01) & (x["p_ref_area"] <= 0.0625) ].copy() 

In [ ]:
# [AREA] Lake size between > 250² m²
cat_sup_250x250_m2=f'> 250² m²'
def _filtered_sup_250x250_m2(x):
    return x[x["p_ref_area"] > 0.0625].copy() 

In [ ]:
# [ALL] Lake size between > 1000² m²
cat_sup_1000x1000_m2=f'> 1000² m²'
def _filtered_sup_1000x1000_m2(x):
    return x[x["p_ref_area"] > 1].copy() 

In [ ]:
# [WSE] Lake size between > 250² m² and < 1 km²
cat_250x250_to_1000x1000_m2=f'> 250² and < 1000² m²'
def _filtered_250x250_to_1000x1000_m2(x):
    return x[(x["p_ref_area"] > 0.0625) & (x["p_ref_area"] <= 1) ].copy() 

In [ ]:
# [ALL] Filter bitflag
def _filter_bitwise(x, bitwise_filter):
    return x.drop(x[x.qual_f_b.apply(lambda y: bool((y >> bitwise_filter) & 1))].index).copy()

In [ ]:
# [ALL] Estimate statistics
def _stats(x, case='area_total', category="category"):
    print("Lake size {}".format(category))
    if case == 'area_total':
        x['case'] = x['area_total']*100 # %
    else:
        x['case'] = x['wse']*100 # cm
    return _p50_p68(x['case'].values), _nb_matchups(x), _nb_pld(x)

In [ ]:
# [ALL] Statistics by lake size requirements 
def _stats_by_lake_size(x, case='area_total'):
    if case == 'area_total':  
        print("STATS AREA")
        # Filter by lake size
        df_100x100_to_250x250_m2 = _filtered_100x100_to_250x250_m2(x)
        df_sup_250x250_m2 = _filtered_sup_250x250_m2(x)
        df_sup_1000x1000_m2 = _filtered_sup_1000x1000_m2(x)
        # Stats by lake size
        stats_no_filtering_1 = _stats(df_100x100_to_250x250_m2, category=cat_100x100_to_250x250_m2)
        stats_no_filtering_2 = _stats(df_sup_250x250_m2, category=cat_sup_250x250_m2)
        stats_no_filtering_3 = _stats(df_sup_1000x1000_m2, category=cat_sup_1000x1000_m2)
    else:
        print("STATS WSE")
        # Filter by lake size
        df_250x250_to_1000x1000_m2_m2 = _filtered_250x250_to_1000x1000_m2(x)
        df_sup_1000x1000_m2 = _filtered_sup_1000x1000_m2(x)
        # Stats by lake size
        stats_no_filtering_1 = _stats(df_250x250_to_1000x1000_m2_m2, 'wse', category=cat_250x250_to_1000x1000_m2)
        stats_no_filtering_2 = _stats(df_sup_1000x1000_m2, 'wse', category=cat_sup_1000x1000_m2)

In [ ]:
# [ALL] Return CDF of absolute values
def _cdf(arr):
    arr = np.abs(arr)
    arr = np.sort(arr[~np.isnan(arr)])
    if len(arr) == 0:
        return arr, arr
    return arr, np.linspace(0, 1, len(arr), endpoint=False)

In [ ]:
# [ALL] Return CDF of relative values
def _cdf_relative(arr):
    arr = np.sort(arr[~np.isnan(arr)])
    if len(arr) == 0:
        return arr, arr
    return arr, np.linspace(0, 1, len(arr), endpoint=False)

In [ ]:
# [ALL] Calculate area in km² from length in metter
def ls_m2km2(length_m):
    area_km2 = (length_m ** 2)/ 1_000_000
    return area_km2

<a id="global-stats"></a>
## Global statistics

<a id="stats-area"></a>
### 1°) Stats AREA

In [ ]:
df_diff = pd.read_csv(os.path.join(input_dir, "diff_area.csv"), sep=";") 
df_diff_area = df_diff.copy()
df_diff.head()

In [ ]:
len(df_diff)

#### No filtering

In [ ]:
df_subset = df_diff.copy()
_nb_matchups(df_subset)
_nb_pld(df_subset)
_stats_by_lake_size(df_subset)

#### partial_f = 0

In [ ]:
df_subset = df_diff[df_diff["partial_f"] == 0].copy()
_nb_matchups(df_subset)
_nb_pld(df_subset)
_stats_by_lake_size(df_subset)

#### partial_f = 0 + ice_f = 0,1

In [ ]:
df_subset = df_diff[
    (df_diff["partial_f"] == 0) &
    (df_diff["ice_f"].isin([0, 1])).copy()
]
_nb_matchups(df_subset)
_nb_pld(df_subset)
_stats_by_lake_size(df_subset)

#### partial_f = 0 + ice_f = 0

In [ ]:
df_subset = df_diff[(df_diff["partial_f"] == 0) & (df_diff["ice_f"] == 0)].copy()
_nb_matchups(df_subset)
_nb_pld(df_subset)
_stats_by_lake_size(df_subset)

#### partial_f = 0 + ice_f = 0 + quality_f=0,1,2

In [ ]:
df_subset = df_diff[
    (df_diff["partial_f"] == 0) &
    (df_diff["ice_f"] == 0) &
    (df_diff["quality_f"].isin([0, 1, 2])).copy()
]
_nb_matchups(df_subset)
_nb_pld(df_subset)
_stats_by_lake_size(df_subset)

#### partial_f = 0 + ice_f = 0 + quality_f=0,1

In [ ]:
df_subset = df_diff[
    (df_diff["partial_f"] == 0) &
    (df_diff["ice_f"] == 0) &
    (df_diff["quality_f"].isin([0, 1])).copy()
]
_nb_matchups(df_subset)
_nb_pld(df_subset)
_stats_by_lake_size(df_subset)

#### partial_f = 0 + ice_f = 0 + quality_f=0

In [ ]:
df_subset = df_diff[(df_diff["partial_f"] == 0) & (df_diff["ice_f"] == 0) & (df_diff["quality_f"] == 0)].copy()
_nb_matchups(df_subset)
_nb_pld(df_subset)
_stats_by_lake_size(df_subset)

#### partial_f = 0 + ice_f = 0 + quality_f=0,1 + wse_std_suspect

In [ ]:
df_subset = df_diff[
    (df_diff["partial_f"] == 0) &
    (df_diff["ice_f"] == 0) &
    (df_diff["quality_f"].isin([0, 1])).copy()
]
_nb_matchups(df_subset)
df_subset_bitwise = _filter_bitwise(df_subset, 5)
_nb_matchups(df_subset_bitwise)
_nb_pld(df_subset_bitwise)
_stats_by_lake_size(df_subset_bitwise)

#### partial_f = 0 + ice_f = 0 + quality_f=0,1 + water_false_detection_rate_suspect

In [ ]:
df_subset = df_diff[
    (df_diff["partial_f"] == 0) &
    (df_diff["ice_f"] == 0) &
    (df_diff["quality_f"].isin([0, 1])).copy()
]
_nb_matchups(df_subset)
df_subset_bitwise = _filter_bitwise(df_subset, 11)
_nb_matchups(df_subset_bitwise)
_nb_pld(df_subset_bitwise)
_stats_by_lake_size(df_subset_bitwise)

#### partial_f = 0 + ice_f = 0 + quality_f=0,1 + xovr_cal_suspect

In [ ]:
df_subset = df_diff[
    (df_diff["partial_f"] == 0) &
    (df_diff["ice_f"] == 0) &
    (df_diff["quality_f"].isin([0, 1])).copy()
]
_nb_matchups(df_subset)
df_subset_bitwise = _filter_bitwise(df_subset, 8)
_nb_matchups(df_subset_bitwise)
_nb_pld(df_subset_bitwise)
_stats_by_lake_size(df_subset_bitwise)

#### partial_f = 0 + ice_f = 0 + quality_f=0,1 + diff_from_pld_area_suspect

In [ ]:
df_subset = df_diff[
    (df_diff["partial_f"] == 0) &
    (df_diff["ice_f"] == 0) &
    (df_diff["quality_f"].isin([0, 1])).copy()
]
_nb_matchups(df_subset)
df_subset_bitwise = _filter_bitwise(df_subset, 6)
_nb_matchups(df_subset_bitwise)
_nb_pld(df_subset_bitwise)
_stats_by_lake_size(df_subset_bitwise)

#### partial_f = 0 + ice_f = 0 + quality_f=0,1 + geolocation_cal_suspect

In [ ]:
df_subset = df_diff[
    (df_diff["partial_f"] == 0) &
    (df_diff["ice_f"] == 0) &
    (df_diff["quality_f"].isin([0, 1])).copy()
]
_nb_matchups(df_subset)
df_subset_bitwise = _filter_bitwise(df_subset, 1)
_nb_matchups(df_subset_bitwise)
_nb_pld(df_subset_bitwise)
_stats_by_lake_size(df_subset_bitwise)

#### partial_f = 0 + ice_f = 0 + quality_f=0,1 + no_prior_suspect

In [ ]:
df_subset = df_diff[
    (df_diff["partial_f"] == 0) &
    (df_diff["ice_f"] == 0) &
    (df_diff["quality_f"].isin([0, 1])).copy()
]
_nb_matchups(df_subset)
df_subset_bitwise = _filter_bitwise(df_subset, 10)
_nb_matchups(df_subset_bitwise)
_nb_pld(df_subset_bitwise)
_stats_by_lake_size(df_subset_bitwise)

#### partial_f = 0 + ice_f = 0 + quality_f=0,1 + classification_qual_suspect

In [ ]:
df_subset = df_diff[
    (df_diff["partial_f"] == 0) &
    (df_diff["ice_f"] == 0) &
    (df_diff["quality_f"].isin([0, 1])).copy()
]
_nb_matchups(df_subset)
df_subset_bitwise = _filter_bitwise(df_subset, 0)
_nb_matchups(df_subset_bitwise)
_nb_pld(df_subset_bitwise)
_stats_by_lake_size(df_subset_bitwise)

#### partial_f = 0 + ice_f = 0 + quality_f=0,1 + few_open_water_suspect

In [ ]:
df_subset = df_diff[
    (df_diff["partial_f"] == 0) &
    (df_diff["ice_f"] == 0) &
    (df_diff["quality_f"].isin([0, 1])).copy()
]
_nb_matchups(df_subset)
df_subset_bitwise = _filter_bitwise(df_subset, 3)
_nb_matchups(df_subset_bitwise)
_nb_pld(df_subset_bitwise)
_stats_by_lake_size(df_subset_bitwise)

#### PLD lake count

In [ ]:
df_filtered = df_diff[df_diff['_fuid'].str.startswith(('PleiadesXS_TP:'))]
print("Nb observations {}".format(len(df_filtered['_fuid'])))
print("Nb observations {}".format(df_filtered['_fuid'].str.split(':').str[1].nunique()))

In [ ]:
df_filtered = df_diff[df_diff['_fuid'].str.startswith(('SLUM_TP:'))]
print("Nb observations {}".format(len(df_filtered['_fuid'])))
print("Nb observations {}".format(df_filtered['_fuid'].str.split(':').str[1].nunique()))

In [ ]:
df_filtered = df_diff[df_diff['_fuid'].str.startswith(('RCM_TP:'))]
print("Nb observations {}".format(len(df_filtered['_fuid'])))
print("Nb observations {}".format(df_filtered['_fuid'].str.split(':').str[1].nunique()))

<a id="stats-wse"></a>
### 2°) Stats WSE

In [ ]:
df_diff = pd.read_csv(os.path.join(input_dir, "diff_wse.csv"), sep=";")  
df_diff_wse = df_diff.copy()
df_diff.head()

#### No filtering

In [ ]:
df_subset = df_diff.copy()
_nb_matchups(df_subset)
_nb_pld(df_subset)
_stats_by_lake_size(df_subset, 'wse')

#### ice_f = 0,1

In [ ]:
df_subset = df_diff[df_diff["ice_f"].isin([0, 1])].copy()
_nb_matchups(df_subset)
_nb_pld(df_subset)
_stats_by_lake_size(df_subset, 'wse')

#### ice_f = 0

In [ ]:
df_subset = df_diff[df_diff["ice_f"] == 0].copy()
_nb_matchups(df_subset)
_nb_pld(df_subset)
_stats_by_lake_size(df_subset, 'wse')

#### ice_f = 0 + partial_f = 0

In [ ]:
df_subset = df_diff[(df_diff["ice_f"] == 0) & (df_diff["partial_f"] == 0)].copy()
_nb_matchups(df_subset)
_nb_pld(df_subset)
_stats_by_lake_size(df_subset, 'wse')

#### ice_f = 0 + partial_f = 0 + quality_f=0,1,2

In [ ]:
df_subset = df_diff[
    (df_diff["ice_f"] == 0) &
    (df_diff["partial_f"] == 0) &
    (df_diff["quality_f"].isin([0, 1, 2]))
].copy()
_nb_matchups(df_subset)
_nb_pld(df_subset)
_stats_by_lake_size(df_subset, 'wse')

#### ice_f = 0 + partial_f = 0 + quality_f=0,1

In [ ]:
df_subset = df_diff[
    (df_diff["ice_f"] == 0) &
    (df_diff["partial_f"] == 0) &
    (df_diff["quality_f"].isin([0, 1]))
].copy()
_nb_matchups(df_subset)
_nb_pld(df_subset)
_stats_by_lake_size(df_subset, 'wse')

#### ice_f = 0 + partial_f = 0 + quality_f=0

In [ ]:
df_subset = df_diff[
    (df_diff["ice_f"] == 0) &
    (df_diff["partial_f"] == 0) &
    (df_diff["quality_f"] == 0)
].copy()
_nb_matchups(df_subset)
_nb_pld(df_subset)
_stats_by_lake_size(df_subset, 'wse')

# Plots

#### Functions for plots

In [ ]:
def stats_df(df, variable, display_tables = False):
    list_sources_df = df["source_name"].unique().tolist()
    df_per_source = df.groupby("source_name").agg(
        n_obs=("_fuid", "count"),   
        n_stations=("_fuid","nunique"),
        n_pld=("lake_id","nunique"),
        source=("source","unique"),
    ).reset_index()
    if display_tables is True:
        display(df_per_source)
    return df_per_source

def group_by_sources(df, variable, by_orbits = False, display_tables = None):
    df["_time"] = pd.to_datetime(df["_time"])
    df = df.sort_values(by="_time")
    df["source"] = df["_fuid"].str.split(":").str[0]
    df['source_name'] = df["source"].astype(str).map(list_sources)
    df["lake_id"] = df["_fuid"].str.split(":").str[1]
    
    df_per_source = stats_df(df, variable, display_tables)

    if by_orbits is True :
        df_calval = df[(df["_time"] >= start_calval) & (df["_time"] <= end_calval)]
        df_per_source_calval = stats_df(df_calval, variable, display_tables)
        df_science = df[(df["_time"] >= start_science)]
        df_per_source_science = stats_df(df_science, variable, display_tables)

    return df, df_per_source

def create_geodataframe(df):
    print(f"Initial length of the dataframe: {len(df)}")
    df["wkb_geometry"] = df["wkb_geometry"].apply(wkt.loads)
    gdf = gpd.GeoDataFrame(df, geometry="wkb_geometry", crs="EPSG:4326")
    df = df[df['wkb_geometry'].notna()]
    print(f"Length of the geodataframe: {len(gdf)}")
    return gdf

def prepare_gdf(gdf):
    gdf_per_lake = gdf.groupby("lake_id")[["wkb_geometry","source_name","p_ref_area","source"]].first()
    gdf_per_lake["color"] = gdf_per_lake["source_name"].map(lambda k: style_dict[k]["color"])
    gdf_per_lake["marker"] = gdf_per_lake["source_name"].map(lambda k: style_dict[k]["marker"])
    return gdf_per_lake

def filter_df(df, variable = "wse", filter_obs=None):

    if variable == "wse" :
        filters = {
            "ice": df["ice_f"].isin([0, 1]),
            "ice_allowed": df["ice_f"].isin([0]),
            "ice_partial_allowed": (df["ice_f"].isin([0])) & (df["partial_f"].isin([0])),
            "ice_partial_quality_012": (df["ice_f"].isin([0])) & (df["partial_f"].isin([0])) & (df["quality_f"].isin([0, 1, 2])),
            "ice_partial_quality_01": (df["ice_f"].isin([0])) & (df["partial_f"].isin([0])) & (df["quality_f"].isin([0, 1])),
            "ice_partial_quality_allowed": (df["ice_f"].isin([0])) & (df["partial_f"].isin([0])) & (df["quality_f"].isin([0])),
        }
    else:
        filters = {
            "partial_allowed": df["partial_f"].isin([0]),
            "partial_ice_allowed": (df["partial_f"].isin([0])) & (df["ice_f"].isin([0])),
            "partial_ice_quality_012": (df["partial_f"].isin([0])) & (df["ice_f"].isin([0])) & (df["quality_f"].isin([0, 1, 2])),
            "partial_ice_quality_01": (df["partial_f"].isin([0])) & (df["ice_f"].isin([0])) & (df["quality_f"].isin([0, 1])),
            "partial_ice_quality_allowed": (df["partial_f"].isin([0])) & (df["ice_f"].isin([0])) & (df["quality_f"].isin([0])),
        }

    if filter_obs is not None:
        df = df[filters[filter_obs]]

    return df

def prepare_df(df, variable = "wse", area_threshold = None, area_limits = None, filter_obs = None, by_orbits = False, display_tables = False):
    # Filter sizes
    if area_threshold is not None :
        print(f"\nInitial dataframe length before size limit: {len(df)}")
        df = df[df.p_ref_area > ls_m2km2(area_threshold)]
        print(f"Dataframe length after size limit: {len(df)}")
    elif area_limits is not None :
        print(f"\nInitial dataframe length before size limit: {len(df)}")
        df = df[(df.p_ref_area > ls_m2km2(area_limits[0])) & (df.p_ref_area < ls_m2km2(area_limits[1]))]
        print(f"Dataframe length after size limit: {len(df)}")

    # Filter dataframe
    df = filter_df(df, variable, filter_obs)
    
    # Group by source name
    df, df_per_source = group_by_sources(df, variable, by_orbits, display_tables)

    # Extract geometries and create a geodataframe
    gdf = create_geodataframe(df)
    
    # Regroup by lakes and prepare for plots
    gdf_per_lake = prepare_gdf(gdf)
    
    return df_per_source, gdf, gdf_per_lake

# Retrieve polygons from URL
def extract_gdf_from_url(url):
    r = requests.get(url, verify=False)
    r.raise_for_status()

    gdf = gpd.GeoDataFrame.from_features(r.json()["features"])
    gdf = gdf[gdf.bounds.maxy > -60]  # crop Antarctica

    return gdf

def prepare_figure(size = (20, 15)):
    fig = plt.figure(figsize=size)

    gs = fig.add_gridspec(
        nrows=2,
        ncols=2,
        height_ratios=[2.2, 0.8],
        hspace=0.0,
        wspace=0.3
    )
    ax_map = fig.add_subplot(gs[0, :])
    ax_sources = fig.add_subplot(gs[1, 0])
    ax_graph = fig.add_subplot(gs[1, 1])

    # Add countries as map background
    url_countries = "https://raw.githubusercontent.com/johan/world.geo.json/master/countries.geo.json"
    world = extract_gdf_from_url(url_countries)
    world.plot(
        ax=ax_map,
        color="0.92",
        edgecolor="0.7",
        linewidth=0.4,
        zorder=0
    )
    
    return fig, ax_map, ax_sources, ax_graph

def map_truth_lakes(ax, gdf_per_lake):
    
    gdf_plot = gdf_per_lake.set_geometry("wkb_geometry")
    for source_name in style_dict:
        group = gdf_plot[gdf_plot["source_name"] == source_name]
        if group.empty:
            continue

        group.plot(
            ax=ax,
            markersize=25,
            marker=style_dict[source_name]["marker"],
            color=style_dict[source_name]["color"],
            edgecolor="black",
            linewidth=0.3,
            alpha=0.9,
            label=source_name,
            zorder=2,
        )

    ax.legend(
        loc="lower left",
        fontsize=12,
        frameon=False,
        facecolor="white",
        edgecolor="none",
        bbox_to_anchor = (0.03,0),
        framealpha=1,
        ncol=1,
    )
    
    ax.set_axis_off()

    return ax
    
def stats_by_sources(df_per_source):
    # Prepare the dataframe by source 
    order = list(style_dict.keys())[::-1]
    df_plot = (
        df_per_source
        .set_index("source_name")
        .reindex(
            [s for s in order
             if s in df_per_source["source_name"].values]
        )
    )
    if not df_plot.index.isin(list_sources_area).all():
        order = list(style_dict_wse.keys())[::-1]
        df_plot = (
            df_per_source
            .set_index("source_name")
            .reindex(order, fill_value=0)
        )


    # Retrieve statistics and colors
    nlakes = df_plot["n_pld"]     
    nmatchups = df_plot["n_obs"]
    colors = [style_dict[s]["color"] for s in df_plot.index]

    return df_plot, nlakes, colors, nmatchups

def bar_plot_sources(ax, df_per_source, matchups=False, letter=False):
    df_plot, nlakes, colors, nmatchups = stats_by_sources(df_per_source)

    # Generate bars
    if matchups is False :
        var = nlakes
        title = "Number of lakes"
    else :
        var = nmatchups
        title = "Number of matchups"
        
    source_pct = (var / var.sum() * 100)
    bars = ax.barh(
        source_pct.index,
        var.values,
        color=colors,
        edgecolor="none"
    )

    # Generate labels ans annotations
    for bar, pct in zip(bars, source_pct.values):
        width = bar.get_width()
        ax.text(
            width + var.max() * 0.02,
            bar.get_y() + bar.get_height()/2,
            f"{pct:.2f}%",
            va="center",
            fontsize=12
        )

    # Format
    ax.set_xlabel(title)
    ax.set_ylabel("")
    ax.set_xlim(0, var.max() * 1.2)
    ax.grid(
        axis="x",
        alpha=0.15,
        linestyle="--"
    )
    ax.tick_params(axis="y", pad=8, labelsize=7)
    # Add letter for figure subplot
    if letter :
        ax.text(
            -0.05, 1.05,
            "(b)",
            transform=ax_sources.transAxes,
            fontsize=14,
            # fontweight="bold"
        )

    return ax
  

def histo_cdf_size_lakes(ax, gdf, matchups = False, letter=False):

    areas = np.sort(gdf["p_ref_area"].values)
    cdf = np.arange(1, len(areas) + 1) / len(areas)

    # Logarithm histogram
    bins_hist = np.logspace(
        np.log10(areas.min()),
        np.log10(areas.max()),
        30
    )
    
    ax.hist(
        areas,
        bins=bins_hist,
        color="grey",
        alpha=0.85,
        edgecolor="white",
        linewidth=0.8,
        zorder=5
    )

    ax.set_xscale("log")

    # CDF
    ax_cdf = ax.twinx()

    ax_cdf.plot(
        areas,
        cdf,
        color="#4C72B0",
        lw=1,
        alpha=0.8,
        zorder=50
    )
    if matchups is True :
        ax.set_ylabel("Number of matchups", fontsize=14)
        ax_cdf.set_ylabel("Cumulative fraction of matchups", color="#4C72B0", fontsize=14)
    else:
        ax.set_ylabel("Number of lakes", fontsize=14)
        ax_cdf.set_ylabel("Cumulative fraction of lakes", color="#4C72B0", fontsize=14)
    ax_cdf.set_ylim(0, 1)
    ax_cdf.tick_params(axis='y', colors='#4C72B0')
    ax_cdf.yaxis.label.set_color('#4C72B0')
    ax_cdf.tick_params(axis='both', labelsize=12)

    # Vertical lines at requirement sizes
    x_lines = [0.01, 0.0625, 0.25, 1.0, 4.0]
    labels = [
        r"$100^2$ m$^2$",
        r"$250^2$ m$^2$",
        r"$500^2$ m$^2$",
        r"$1000^2$ m$^2$",
        r"$2000^2$ m$^2$",
    ]

    for x, label in zip(x_lines, labels):
        ax.axvline(
            x,
            color="k",
            linestyle="--",
            linewidth=1.5,
            alpha=0.8,
            zorder=6,
        )
    
        ax.text(
            x,
            1.02,
            label,
            transform=ax.get_xaxis_transform(),
            rotation=45,
            ha="left",
            va="bottom",
            color="k",
            fontsize=12,
            clip_on=False,
        )

    # Format
    ax.xaxis.set_major_locator(LogLocator(base=10))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:g}"))
    ax.set_xlabel("Lake area (km²)")
    yticks = [0.00, 0.16, 0.50, 0.68, 0.84, 1.00]
    ax_cdf.set_yticks(yticks)
    ax_cdf.set_yticklabels([f"{v:.2f}" for v in yticks])
    ax_cdf.grid(True, axis='y', linestyle='--', alpha=0.5)

    # Add letter for figure subplot
    if letter : 
        ax.text(
            -0.05, 1.05,
            "(c)",
            transform=ax_graph.transAxes,
            fontsize=14,
        )

    return ax

def display_save_figure(fig, ax_map, ax_sources, ax_graph, name = None):
    pos_a = ax_map.get_position()
    pos_b = ax_sources.get_position()
    pos_c = ax_graph.get_position()

    ax_sources.set_position([
        pos_b.x0 + 0.05, 
        pos_b.y0,
        pos_b.width,
        pos_b.height
    ])
    for ax in [ax_map, ax_sources, ax_graph] :
        ax.tick_params(axis='both', labelsize=12)
        ax.set_xlabel(ax.get_xlabel(), fontsize=14)
        ax.set_ylabel(ax.get_ylabel(), fontsize=14)

    path_fig = os.path.join( out_dir, f"{name}.png")
    fig.savefig(path_fig, dpi=600, bbox_inches="tight", facecolor="white")
    print(f"Figure saved to : {path_fig}")
    plt.close(fig)

<a id="repartition"></a>
## Plot of repartition

### AREA

In [ ]:
df_diff_area_per_source, gdf_diff_area, gdf_diff_area_per_lake = prepare_df(df_diff_area, variable = "area_total", by_orbits = False, display_tables = False)
fig, ax_map, ax_sources, ax_graph = prepare_figure()
ax_map = map_truth_lakes(ax_map, gdf_diff_area_per_lake)
ax_sources = bar_plot_sources(ax_sources, df_diff_area_per_source)
ax_graph = histo_cdf_size_lakes(ax_graph, gdf_diff_area_per_lake)
for ax, label in zip(fig.axes, "abc"):
    if label == "a":
        ax.text(
        0, 1.0,
        f"({label})",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=16,
        )
    else:
       ax.text(
        -0.12, 1.0,
        f"({label})",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=16,
        ) 
display_save_figure(fig, ax_map, ax_sources, ax_graph, name="area_map_bar_plot_cdf")

### WSE

In [ ]:
df_diff_wse_per_source, gdf_diff_wse, gdf_diff_wse_per_lake = prepare_df(df_diff_wse, variable = "wse", by_orbits = False, display_tables = False)
fig, ax_map, ax_sources, ax_graph = prepare_figure()
ax_map = map_truth_lakes(ax_map, gdf_diff_wse_per_lake)
ax_sources = bar_plot_sources(ax_sources, df_diff_wse_per_source)
ax_graph = histo_cdf_size_lakes(ax_graph, gdf_diff_wse_per_lake)
for ax, label in zip(fig.axes, "abc"):
    if label == "a":
        ax.text(
        0, 1.0,
        f"({label})",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=16,
        )
    else:
       ax.text(
        -0.12, 1.0,
        f"({label})",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=16,
        ) 
display_save_figure(fig, ax_map, ax_sources, ax_graph, name="wse_map_bar_plot_cdf")

<a id="plots-errors"></a>
## Plots of errors and distributions

### AREA

##### Base data (no filtering)

In [ ]:
df_plot = df_diff_area.copy()
print(len(df_plot))
df_plot_area = df_plot.copy()

##### Filtered data (severe filtering: partial_f=0, ice_f=0, quality_f=0)

In [ ]:
df_plot_filtered = df_diff_area[(df_diff_area["partial_f"] == 0) & (df_diff_area["ice_f"] == 0) & (df_diff_area["quality_f"] == 0)].copy()
len(df_plot_filtered)

In [ ]:
df_plot_filtered_area = df_plot_filtered.copy()

In [ ]:
df_diff_area_per_source_filtered, gdf_diff_area_filtered, gdf_diff_area_per_lake_filtered = prepare_df(df_plot_filtered_area, variable = "area_total", by_orbits = False, display_tables = False)

<a id="box-plots-area-altitude"></a>
#### Box plots of AREA error as a function of altitude

In [ ]:
# Lake altitude (p_ref_wse is used as reference)
altitude = np.abs(df_plot["p_ref_wse"])

classes = [
    (-250, 0),
    (0, 250),
    (250, 500),
    (500, 750),
    (750, 1000),
    (1000, 1250),
    (1250, 1500),
    (1500, 1750),
    (1750, 2000),
    (2000, 2250),
    (2250, 2500),
    (2500, 2750),
    (2750, 3000),
    (3000, 3250),
    (3250, 3500),
    (3500, 3750),
    (3750, 4000),
    (4000, 4250),
    (4250, 4500),
]
labels = ["-125", "125", "375", "625", "875", "1125", "1375", "1625", "1875", 
          "2125", "2375", "2625", "2875", "3125", "3375", "3625", "3875", 
          "4125", "4375"]

# Box data for no filtering
box_data = [
    df_plot.loc[
        (altitude >= hmin) & (altitude < hmax),
        "area_total"
    ].dropna()
    for hmin, hmax in classes
]
# Box data for severe filtering
box_data_filtered = [
    df_plot_filtered.loc[
        (altitude >= hmin) & (altitude < hmax),
        "area_total"
    ]
    for hmin, hmax in classes
]

In [ ]:
stats = []
stats_filtered = []
# Base data
for values, label in zip(box_data, labels):
    if len(values) > 0 :
        stats.append({
            "label": label,
            "whislo": np.percentile(values, 16),  # lower whisker
            "q1":     np.percentile(values, 32),  # lower edge
            "med":    np.percentile(values, 50),  # median
            "q3":     np.percentile(values, 68),  # upper edge
            "whishi": np.percentile(values, 84),  # upper whisker
            "fliers": []
        })
    else :
        stats.append({
            "label": label,
            "whislo": 0,
            "q1":     0, 
            "med":    0, 
            "q3":     0,  
            "whishi": 0,  
            "fliers": []
        })
    
# Filtered data
for values, label in zip(box_data_filtered, labels):
    if len(values) > 0 :
        stats_filtered.append({
            "label": label,
            "whislo": np.percentile(values, 16), 
            "q1":     np.percentile(values, 32), 
            "med":    np.percentile(values, 50), 
            "q3":     np.percentile(values, 68),
            "whishi": np.percentile(values, 84),
            "fliers": []
        })
    else:
        stats_filtered.append({
            "label": label,
            "whislo": 0,
            "q1":     0,
            "med":    0,
            "q3":     0,
            "whishi": 0,
            "fliers": []
        })

In [ ]:
fig, axs = plt.subplots(2,1,figsize=(10, 8))

# Graph on the left: No filtering case
bp = axs[0].bxp(
    stats,
    positions = [-125, 125, 375, 625, 875, 1125, 1375, 1625, 1875, 
          2125, 2375, 2625, 2875, 3125, 3375, 3625, 3875, 4125, 4375],
    widths=50, 
    showfliers=False,
    patch_artist=True
)
for box in bp["boxes"]:
    box.set(facecolor="lightblue", alpha=0.7)

axs[0].set_ylabel("Fractional area error (%)")
axs[0].set_title("No filtering")
axs[0].grid(True, axis="y", color="lightgray", linestyle=":", alpha=0.7)
axs[0].grid(True, axis="x", color="lightgray", linestyle=":", alpha=0.7)
axs[0].yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y*100:.0f}"))

# Graph on the right: Severe filtering case
bp = axs[1].bxp(
    stats_filtered,
    positions = [-125, 125, 375, 625, 875, 1125, 1375, 1625, 1875, 
          2125, 2375, 2625, 2875, 3125, 3375, 3625, 3875, 4125, 4375],
    widths=50, 
    showfliers=False,
    patch_artist=True
)
for box in bp["boxes"]:
    box.set(facecolor="lightblue", alpha=0.7)

axs[1].set_xlabel("Altitude (m)")
axs[1].set_ylabel("Fractional area error (%)")
axs[1].set_title("Severe filtering")
axs[1].grid(True, axis="y", color="lightgray", linestyle=":", alpha=0.7)
axs[1].grid(True, axis="x", color="lightgray", linestyle=":", alpha=0.7)
axs[1].yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y*100:.0f}"))

# Set axes limits and scale
for ax in axs:
    ax.set_xlim(-255, 4255)
    ax.set_xticks([-250, 0, 250, 500, 750, 1000, 
                   1250, 1500, 1750, 2000, 2250, 2500, 2750, 3000, 
                   3250, 3500, 3750, 4000, 4250])
    ax.set_xticklabels(["-250", "0", "250", "500", "750", "1000", "1250", "1500",
          "1750", "2000", "2250", "2500", "2750", "3000", "3250", "3500", "3750", "4000", "4250"])
axs[0].set_ylim(-1, 2)
path_fig = os.path.join(out_dir, "ALL_TP_boxplot_area_vs_altitude.png")
plt.savefig(path_fig, dpi=300)
print(f"Figure saved to : {path_fig}")
plt.close()

#### Prepare box plots of AREA error as a function of cross-track

In [ ]:
# Xtrack distance from nadir
xkm = np.abs(df_plot["xtrk_dist"]) / 1000

# Classes
classes = [
    (10, 20),
    (20, 30),
    (30, 40),
    (40, 50),
    (50, 60),
]
labels = ["10", "20", "30", "40", "50"]

# Box data for no filtering
box_data = [
    df_plot.loc[
        (xkm >= xmin) & (xkm < xmax),
        "area_total"
    ].dropna()
    for xmin, xmax in classes
]
# Box data for severe filtering
box_data_filtered = [
    df_plot_filtered.loc[
        (xkm >= xmin) & (xkm < xmax),
        "area_total"
    ].dropna()
    for xmin, xmax in classes
]

In [ ]:
stats = []
stats_filtered = []
# Base data
for values, label in zip(box_data, labels):
    stats.append({
        "label": label,
        "whislo": np.percentile(values, 16),  
        "q1":     np.percentile(values, 32), 
        "med":    np.percentile(values, 50), 
        "q3":     np.percentile(values, 68),
        "whishi": np.percentile(values, 84),
        "fliers": []
    })
    
# Filtered data
for values, label in zip(box_data_filtered, labels):
    stats_filtered.append({
        "label": label,
        "whislo": np.percentile(values, 16),
        "q1":     np.percentile(values, 32),
        "med":    np.percentile(values, 50),
        "q3":     np.percentile(values, 68),
        "whishi": np.percentile(values, 84),
        "fliers": []
    })

#### Prepare CDF of AREA error distribution

##### Base data (no filtering)

In [ ]:
df_plot_lake_size_100x100_to_250x250_m2 = _filtered_100x100_to_250x250_m2(df_diff_area) # > 100² et <= 250² m²
df_plot_lake_size_sup_250x250_m2 = _filtered_sup_250x250_m2(df_diff_area) # > 250² m²
df_plot_lake_size_sup_1000x1000_m2 = _filtered_sup_1000x1000_m2(df_diff_area) # > 1000² m²

##### Filtered Data (severe filtering: partial_f=0 + ice_f=0 + quality_f=0)

In [ ]:
df_plot_partial_f_eq0_ice_f_eq0_quality_f_eq0 = df_diff_area[(df_diff_area["partial_f"] == 0) & (df_diff_area["ice_f"] == 0) & (df_diff_area["quality_f"] == 0)].copy()

In [ ]:
df_plot_lake_size_100x100_to_250x250_m2_filtered = _filtered_100x100_to_250x250_m2(df_plot_partial_f_eq0_ice_f_eq0_quality_f_eq0) # > 100² et <= 250² m²
df_plot_lake_size_sup_250x250_m2_filtered = _filtered_sup_250x250_m2(df_plot_partial_f_eq0_ice_f_eq0_quality_f_eq0) # > 250² m²
df_plot_lake_size_sup_1000x1000_m2_filtered = _filtered_sup_1000x1000_m2(df_plot_partial_f_eq0_ice_f_eq0_quality_f_eq0) # > 1000² m²

<a id="resume_area"></a>
### Characteristics of the measured LakeSP_Prior fractionnal area error without any filtering and severe filtering and lakes distributions

In [ ]:
AREA_color_lake_size_100x100_to_250x250_m2 = "#0072B2"
AREA_color_lake_size_sup_250x250_m2  = "#CC79A7"
AREA_color_lake_size_sup_1000x1000_m2  = "#E69F00"

fig = plt.figure(figsize=(16,16), constrained_layout=True)

gs = fig.add_gridspec(
    nrows=4,
    ncols=2,
)

# No filter
ax_cdf_no_filter = fig.add_subplot(gs[0, 0])
ax_cross_track_no_filter = fig.add_subplot(gs[1, 0])
ax_histo_cdf_no_filter = fig.add_subplot(gs[2, 0])
ax_sources_no_filter = fig.add_subplot(gs[3, 0])

# Filter
ax_cdf_filter = fig.add_subplot(gs[0, 1])
ax_cross_track_filter    = fig.add_subplot(gs[1, 1])
ax_histo_cdf_filter = fig.add_subplot(gs[2, 1])
ax_sources_filter = fig.add_subplot(gs[3, 1])

ax_sources_no_filter = bar_plot_sources(ax_sources_no_filter, df_diff_area_per_source, matchups = True)
ax_histo_cdf_no_filter = histo_cdf_size_lakes(ax_histo_cdf_no_filter, gdf_diff_area, matchups = True)

ax_sources_filter = bar_plot_sources(ax_sources_filter, df_diff_area_per_source_filtered, matchups = True)
ax_histo_cdf_filter = histo_cdf_size_lakes(ax_histo_cdf_filter, gdf_diff_area_filtered, matchups = True)

# whisker no filtering
bp = ax_cross_track_no_filter.bxp(
    stats,
    positions=[15, 25, 35, 45, 55],
    widths=4, # in km
    showfliers=False,
    patch_artist=True
)
for box in bp["boxes"]:
    box.set(facecolor="lightblue", alpha=0.7)

ax_cross_track_no_filter.set_xlabel("Cross-track distance from nadir (km)")
ax_cross_track_no_filter.set_ylabel("Fractional area error (%)")
ax_cross_track_no_filter.grid(True, axis="y", color="lightgray", linestyle=":", alpha=0.7)
ax_cross_track_no_filter.grid(True, axis="x", color="lightgray", linestyle=":", alpha=0.7)
ax_cross_track_no_filter.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y*100:.0f}"))
ax_cross_track_no_filter.set_xlim(5, 65)
ax_cross_track_no_filter.set_xticks([10, 20, 30, 40, 50, 60])
ax_cross_track_no_filter.set_xticklabels(["10", "20", "30", "40", "50", "60"])

# whisker filtering
bp = ax_cross_track_filter.bxp(
    stats_filtered,
    positions=[15, 25, 35, 45, 55],
    widths=4, # in km
    showfliers=False,
    patch_artist=True
)
for box in bp["boxes"]:
    box.set(facecolor="lightblue", alpha=0.7)

ax_cross_track_filter.set_xlabel("Cross-track distance from nadir (km)")
ax_cross_track_filter.set_ylabel("Fractional area error (%)")
ax_cross_track_filter.grid(True, axis="y", color="lightgray", linestyle=":", alpha=0.7)
ax_cross_track_filter.grid(True, axis="x", color="lightgray", linestyle=":", alpha=0.7)
ax_cross_track_filter.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y*100:.0f}"))
ax_cross_track_filter.set_xlim(5, 65)
ax_cross_track_filter.set_xticks([10, 20, 30, 40, 50, 60])
ax_cross_track_filter.set_xticklabels(["10", "20", "30", "40", "50", "60"])
    
# BASE DATA
# Relative values
x, y = _cdf_relative(df_plot_lake_size_100x100_to_250x250_m2['area_total'].values*100);   
ax_cdf_no_filter.plot(x, y, color=AREA_color_lake_size_100x100_to_250x250_m2, label=cat_100x100_to_250x250_m2)

x, y = _cdf_relative(df_plot_lake_size_sup_250x250_m2['area_total'].values*100);   
ax_cdf_no_filter.plot(x, y, color=AREA_color_lake_size_sup_250x250_m2, label=cat_sup_250x250_m2)

x, y = _cdf_relative(df_plot_lake_size_sup_1000x1000_m2['area_total'].values*100);  
ax_cdf_no_filter.plot(x, y, color=AREA_color_lake_size_sup_1000x1000_m2, label=cat_sup_1000x1000_m2)

# Absolute values
x, y = _cdf(df_plot_lake_size_100x100_to_250x250_m2['area_total'].values*100);   
ax_cdf_no_filter.plot(x, y, color=AREA_color_lake_size_100x100_to_250x250_m2, linestyle='--')

x, y = _cdf(df_plot_lake_size_sup_250x250_m2['area_total'].values*100);   
ax_cdf_no_filter.plot(x, y, color=AREA_color_lake_size_sup_250x250_m2, linestyle='--')

x, y = _cdf(df_plot_lake_size_sup_1000x1000_m2['area_total'].values*100);  
ax_cdf_no_filter.plot(x, y, color=AREA_color_lake_size_sup_1000x1000_m2, linestyle='--')


ax_cdf_no_filter.set_xlim(-100, 100)
ax_cdf_no_filter.set_ylim(0, 1)
ax_cdf_no_filter.set_xlabel('Fractional AREA error (%)')
ax_cdf_no_filter.set_ylabel('CDF')
ax_cdf_no_filter.grid(True, which='both', linewidth=0.5)

# Update legend
ax_cdf_no_filter.legend(fontsize=9)
yticks = [0.00, 0.16, 0.50, 0.68, 0.84, 1.00]
ax_cdf_no_filter.set_yticks(yticks)
ax_cdf_no_filter.set_yticklabels([f"{v:.2f}" for v in yticks])
ax_cdf_no_filter.grid(True, axis='y', linestyle='--', alpha=0.5)

# FILTERED DATA
# Relative values
x, y = _cdf_relative(df_plot_lake_size_100x100_to_250x250_m2_filtered['area_total'].values*100);   
ax_cdf_filter.plot(x, y, color=AREA_color_lake_size_100x100_to_250x250_m2, label=cat_100x100_to_250x250_m2)

x, y = _cdf_relative(df_plot_lake_size_sup_250x250_m2_filtered['area_total'].values*100);   
ax_cdf_filter.plot(x, y, color=AREA_color_lake_size_sup_250x250_m2, label=cat_sup_250x250_m2)

x, y = _cdf_relative(df_plot_lake_size_sup_1000x1000_m2_filtered['area_total'].values*100);  
ax_cdf_filter.plot(x, y, color=AREA_color_lake_size_sup_1000x1000_m2, label=cat_sup_1000x1000_m2)

# Absolute values
x, y = _cdf(df_plot_lake_size_100x100_to_250x250_m2_filtered['area_total'].values*100);   
ax_cdf_filter.plot(x, y, color=AREA_color_lake_size_100x100_to_250x250_m2, linestyle='--')

x, y = _cdf(df_plot_lake_size_sup_250x250_m2_filtered['area_total'].values*100);   
ax_cdf_filter.plot(x, y, color=AREA_color_lake_size_sup_250x250_m2, linestyle='--')

x, y = _cdf(df_plot_lake_size_sup_1000x1000_m2_filtered['area_total'].values*100);  
ax_cdf_filter.plot(x, y, color=AREA_color_lake_size_sup_1000x1000_m2, linestyle='--')

ax_cdf_filter.set_xlim(-100, 100)
ax_cdf_filter.set_ylim(0, 1)
ax_cdf_filter.set_xlabel('Fractional AREA error (%)')
ax_cdf_filter.set_ylabel('CDF')
ax_cdf_filter.grid(True, which='both', linewidth=0.5)

# Update légende
ax_cdf_filter.legend(fontsize=9)

# Format
yticks = [0.00, 0.16, 0.50, 0.68, 0.84, 1.00]
ax_cdf_filter.set_yticks(yticks)
ax_cdf_filter.set_yticklabels([f"{v:.2f}" for v in yticks])
ax_cdf_filter.grid(True, axis='y', linestyle='--', alpha=0.5)

for ax, label in zip(fig.axes, "acegbdfh"):
    ax.text(
    -0.12, 1.05,
    f"({label})",
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=16,
    )
    ax.tick_params(axis='both', labelsize=12)
    ax.set_xlabel(ax.get_xlabel(), fontsize=14)
    ax.set_ylabel(ax.get_ylabel(), fontsize=14)
    
ax_cdf_no_filter.set_title("No filtering", fontsize=16, pad = 15, fontweight="medium")
ax_cdf_filter.set_title("Severe filtering", fontsize=16, pad= 15, fontweight="medium")

path_fig = os.path.join(out_dir, "area_cdf_box_histo_sources.png")
plt.savefig(path_fig, dpi=300)
print(f"Figure saved to : {path_fig}")
plt.close()

### WSE

##### Base data (no filtering)

In [ ]:
df_plot = df_diff_wse.copy()
len(df_plot)

##### Filtered data (severe filtering: partial_f=0, ice_f=0, quality_f=0)

In [ ]:
df_plot_filtered = df_diff_wse[(df_diff_wse["partial_f"] == 0) & (df_diff_wse["ice_f"] == 0) & (df_diff_wse["quality_f"] == 0)].copy()
len(df_plot_filtered)

In [ ]:
df_plot_filtered_wse = df_plot_filtered.copy()

<a id="box-plots-wse-altitude"></a>
#### Box plots of WSE error as a function of altitude

In [ ]:
# Lake altitude (p_ref_wse is used as reference)
altitude = np.abs(df_plot["p_ref_wse"])

classes = [
    (0, 250),
    (250, 500),
    (500, 750),
    (750, 1000),
    (1000, 1250),
    (1250, 1500),
    (1500, 1750),
    (1750, 2000),
    (2000, 2250),
    (2250, 2500),
    (2500, 2750),
    (2750, 3000),
    (3000, 3250),
    (3250, 3500),
    (3500, 3750),
    (3750, 4000),
    (4000, 4250),
    (4250, 4500),
    (4500, 4750),
]
labels = ["125", "375", "625", "875", "1125", "1375", "1625", "1875", 
          "2125", "2375", "2625", "2875", "3125", "3375", "3625", "3875",
         "4125", "4375", "4625"]

# Box data for no filtering
box_data = [
    df_plot.loc[
        (altitude >= hmin) & (altitude < hmax),
        "wse"
    ].dropna()
    for hmin, hmax in classes
]
# Box data for severe filtering
box_data_filtered = [
    df_plot_filtered.loc[
        (altitude >= hmin) & (altitude < hmax),
        "wse"
    ]
    for hmin, hmax in classes
]

In [ ]:
stats = []
stats_filtered = []
# Base data
for values, label in zip(box_data, labels):
    if len(values) > 0 :
        stats.append({
            "label": label,
            "whislo": np.percentile(values, 16),
            "q1":     np.percentile(values, 32),
            "med":    np.percentile(values, 50),
            "q3":     np.percentile(values, 68), 
            "whishi": np.percentile(values, 84),
            "fliers": []
        })
    else :
        stats.append({
            "label": label,
            "whislo": 0,
            "q1":     0,
            "med":    0,
            "q3":     0,
            "whishi": 0,
            "fliers": []
        })
    
# Filtered data
for values, label in zip(box_data_filtered, labels):
    if len(values) > 0 :
        stats_filtered.append({
            "label": label,
            "whislo": np.percentile(values, 16), 
            "q1":     np.percentile(values, 32), 
            "med":    np.percentile(values, 50), 
            "q3":     np.percentile(values, 68),
            "whishi": np.percentile(values, 84), 
            "fliers": []
        })
    else:
        stats_filtered.append({
            "label": label,
            "whislo": 0,
            "q1":     0,
            "med":    0, 
            "q3":     0,
            "whishi": 0,
            "fliers": []
        })

In [ ]:
fig, axs = plt.subplots(2,1,figsize=(10, 8))

# Graph on the left: no filtering case
bp = axs[0].bxp(
    stats,
    positions = [125, 375, 625, 875, 1125, 1375, 1625, 1875,
          2125, 2375, 2625, 2875, 3125, 3375, 3625, 3875,
         4125, 4375, 4625],
    widths=40, 
    showfliers=False,
    patch_artist=True
)
for box in bp["boxes"]:
    box.set(facecolor="lightblue", alpha=0.7)

axs[0].set_ylabel("WSE error (cm)")
axs[0].set_title("No filtering")
axs[0].grid(True, axis="y", color="lightgray", linestyle=":", alpha=0.7)
axs[0].grid(True, axis="x", color="lightgray", linestyle=":", alpha=0.7)
axs[0].yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y*100:.0f}"))

# Graph on the right: severe filtering case
bp = axs[1].bxp(
    stats_filtered,
    positions = [125, 375, 625, 875, 1125, 1375, 1625, 1875,
          2125, 2375, 2625, 2875, 3125, 3375, 3625, 3875,
         4125, 4375, 4625],
    widths=40, 
    showfliers=False,
    patch_artist=True
)
for box in bp["boxes"]:
    box.set(facecolor="lightblue", alpha=0.7)

axs[1].set_xlabel("Altitude (m)")
axs[1].set_ylabel("WSE error (cm)")
axs[1].set_title("Severe filtering")
axs[1].grid(True, axis="y", color="lightgray", linestyle=":", alpha=0.7)
axs[1].grid(True, axis="x", color="lightgray", linestyle=":", alpha=0.7)
axs[1].yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y*100:.0f}"))

# Set axes limits and scale
for ax in axs:
    ax.set_xlim(50, 4650)
    ax.set_xticks([0, 250, 500, 750, 1000, 
                   1250, 1500, 1750, 2000, 
                   2250, 2500, 2750, 3000, 
                   3250, 3500, 3750, 4000,
                   4250, 4500, 4750])
    ax.set_xticklabels(["0", "250", "500", "750", "1000", "1250", "1500", "1750", 
          "2000", "2250", "2500", "2750", "3000", "3250", "3500", "3750",
         "4000", "4250", "4500", "4750"])
axs[0].set_ylim(-1, 2)

plt.savefig(os.path.join(out_dir, "ALL_TP_boxplot_wse_vs_altitude.png"), dpi=300)
plt.close()

#### Prepare box plots of WSE error as a function of cross-track

In [ ]:
# Xtrck distance from nadir
xkm = np.abs(df_plot["xtrk_dist"]) / 1000

# Classes
classes = [
    (10, 20),
    (20, 30),
    (30, 40),
    (40, 50),
    (50, 60),
]
labels = ["10", "20", "30", "40", "50"]

# Box data for no filtering
box_data = [
    df_plot.loc[
        (xkm >= xmin) & (xkm < xmax),
        "wse"
    ].dropna()
    for xmin, xmax in classes
]
# Box data for severe filtering
box_data_filtered = [
    df_plot_filtered.loc[
        (xkm >= xmin) & (xkm < xmax),
        "wse"
    ].dropna()
    for xmin, xmax in classes
]

In [ ]:
stats = []
stats_filtered = []

# Base data
for values, label in zip(box_data, labels):
    stats.append({
        "label": label,
        "whislo": np.percentile(values, 16), 
        "q1":     np.percentile(values, 32),
        "med":    np.percentile(values, 50),
        "q3":     np.percentile(values, 68),
        "whishi": np.percentile(values, 84), 
        "fliers": []
    })

# Filtered data
for values, label in zip(box_data_filtered, labels):
    stats_filtered.append({
        "label": label,
        "whislo": np.percentile(values, 16), 
        "q1":     np.percentile(values, 32), 
        "med":    np.percentile(values, 50), 
        "q3":     np.percentile(values, 68), 
        "whishi": np.percentile(values, 84), 
        "fliers": []
    })

#### Prepare CDF of WSE error distribution

##### Base Data (no filtering)

In [ ]:
df_plot_lake_size_250x250_to_1000x1000_m2 = _filtered_250x250_to_1000x1000_m2(df_diff_wse) # > 250² et <= 1000² m²
df_plot_lake_size_sup_1000x1000_m2 = _filtered_sup_1000x1000_m2(df_diff_wse) # > 1000² m²

##### Filtered Data (severe filtering: partial_f=0 + ice_f=0 + quality_f=0)

In [ ]:
df_plot_partial_f_eq0_ice_f_eq0_quality_f_eq0 = df_diff_wse[(df_diff_wse["partial_f"] == 0) & (df_diff_wse["ice_f"] == 0) & (df_diff_wse["quality_f"] == 0)].copy()

In [ ]:
df_plot_lake_size_250x250_to_1000x1000_m2_filtered = _filtered_250x250_to_1000x1000_m2(df_plot_partial_f_eq0_ice_f_eq0_quality_f_eq0) # > 250² et <= 1000² m²
df_plot_lake_size_sup_1000x1000_m2_filtered = _filtered_sup_1000x1000_m2(df_plot_partial_f_eq0_ice_f_eq0_quality_f_eq0) # > 1000² m²

In [ ]:
df_diff_wse_per_source_filtered, gdf_diff_wse_filtered, gdf_diff_wse_per_lake_filtered = prepare_df(df_plot_filtered_wse, variable = "wse", by_orbits = False, display_tables = False)

<a id="resume_wse"></a>
### Characteristics of the measured LakeSP_Prior WSE error without any filtering and severe filtering and lakes distributions

In [ ]:
WSE_color_lake_size_250x250_to_1000x1000_m2 = "#0072B2"
WSE_color_lake_size_sup_1000x1000_m2  = "#E69F00"

fig = plt.figure(figsize=(18,18), constrained_layout=True)

gs = fig.add_gridspec(
    nrows=4,
    ncols=2,
)

# No filter
ax_cdf_no_filter = fig.add_subplot(gs[0, 0])
ax_cross_track_no_filter = fig.add_subplot(gs[1, 0])
ax_histo_cdf_no_filter = fig.add_subplot(gs[2, 0])
ax_sources_no_filter = fig.add_subplot(gs[3, 0])

# Filter
ax_cdf_filter = fig.add_subplot(gs[0, 1])
ax_cross_track_filter    = fig.add_subplot(gs[1, 1])
ax_histo_cdf_filter = fig.add_subplot(gs[2, 1])
ax_sources_filter = fig.add_subplot(gs[3, 1])

ax_sources_no_filter = bar_plot_sources(ax_sources_no_filter, df_diff_wse_per_source, matchups = True)
ax_histo_cdf_no_filter = histo_cdf_size_lakes(ax_histo_cdf_no_filter, gdf_diff_wse, matchups = True)

ax_sources_filter = bar_plot_sources(ax_sources_filter, df_diff_wse_per_source_filtered, matchups = True)
ax_histo_cdf_filter = histo_cdf_size_lakes(ax_histo_cdf_filter, gdf_diff_wse_filtered, matchups = True)

# whisker no filtering
bp = ax_cross_track_no_filter.bxp(
    stats,
    positions=[15, 25, 35, 45, 55],
    widths=4, 
    showfliers=False,
    patch_artist=True
)
for box in bp["boxes"]:
    box.set(facecolor="lightblue", alpha=0.7)

ax_cross_track_no_filter.set_xlabel("Cross-track distance from nadir (km)")
ax_cross_track_no_filter.set_ylabel("WSE error (cm)")
ax_cross_track_no_filter.grid(True, axis="y", color="lightgray", linestyle=":", alpha=0.7)
ax_cross_track_no_filter.grid(True, axis="x", color="lightgray", linestyle=":", alpha=0.7)
ax_cross_track_no_filter.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y*100:.0f}"))
ax_cross_track_no_filter.set_xlim(5, 65)
ax_cross_track_no_filter.set_ylim(-0.3, 1.5)
ax_cross_track_no_filter.set_xticks([10, 20, 30, 40, 50, 60])
ax_cross_track_no_filter.set_xticklabels(["10", "20", "30", "40", "50", "60"])

# whisker filtering
bp = ax_cross_track_filter.bxp(
    stats_filtered,
    positions=[15, 25, 35, 45, 55],
    widths=4, 
    showfliers=False,
    patch_artist=True
)
for box in bp["boxes"]:
    box.set(facecolor="lightblue", alpha=0.7)

ax_cross_track_filter.set_xlabel("Cross-track distance from nadir (km)")
ax_cross_track_filter.set_ylabel("WSE error (cm)")
ax_cross_track_filter.grid(True, axis="y", color="lightgray", linestyle=":", alpha=0.7)
ax_cross_track_filter.grid(True, axis="x", color="lightgray", linestyle=":", alpha=0.7)
ax_cross_track_filter.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y*100:.0f}"))
ax_cross_track_filter.set_xlim(5, 65)
ax_cross_track_filter.set_xticks([10, 20, 30, 40, 50, 60])
ax_cross_track_filter.set_xticklabels(["10", "20", "30", "40", "50", "60"])

# BASE DATA
# Relative values
x, y = _cdf_relative(df_plot_lake_size_250x250_to_1000x1000_m2['wse'].values*100);   
ax_cdf_no_filter.plot(x, y, color=WSE_color_lake_size_250x250_to_1000x1000_m2, label=cat_250x250_to_1000x1000_m2)

x, y = _cdf_relative(df_plot_lake_size_sup_1000x1000_m2['wse'].values*100);  
ax_cdf_no_filter.plot(x, y, color=WSE_color_lake_size_sup_1000x1000_m2, label=cat_sup_1000x1000_m2)

# Absolute values (cm)
x, y = _cdf(df_plot_lake_size_250x250_to_1000x1000_m2['wse'].values*100);   
ax_cdf_no_filter.plot(x, y, color=WSE_color_lake_size_250x250_to_1000x1000_m2, linestyle='--')

x, y = _cdf(df_plot_lake_size_sup_1000x1000_m2['wse'].values*100);  
ax_cdf_no_filter.plot(x, y, color=WSE_color_lake_size_sup_1000x1000_m2, linestyle='--')


ax_cdf_no_filter.set_xlim(-100, 100)
ax_cdf_no_filter.set_ylim(0, 1)
ax_cdf_no_filter.set_xlabel("WSE error (cm)")
ax_cdf_no_filter.set_ylabel('CDF')
ax_cdf_no_filter.grid(True, which='both', linewidth=0.5)

# Update legend
ax_cdf_no_filter.legend(fontsize=9)
yticks = [0.00, 0.16, 0.50, 0.68, 0.84, 1.00]
ax_cdf_no_filter.set_yticks(yticks)
ax_cdf_no_filter.set_yticklabels([f"{v:.2f}" for v in yticks])
ax_cdf_no_filter.grid(True, axis='y', linestyle='--', alpha=0.5)

# FILTERED DATA
# Relative values (cm)
x, y = _cdf_relative(df_plot_lake_size_250x250_to_1000x1000_m2_filtered['wse'].values*100);   
ax_cdf_filter.plot(x, y, color=WSE_color_lake_size_250x250_to_1000x1000_m2, label=cat_250x250_to_1000x1000_m2)

x, y = _cdf_relative(df_plot_lake_size_sup_1000x1000_m2_filtered['wse'].values*100);  
ax_cdf_filter.plot(x, y, color=WSE_color_lake_size_sup_1000x1000_m2, label=cat_sup_1000x1000_m2)

# Absolute values (cm)
x, y = _cdf(df_plot_lake_size_250x250_to_1000x1000_m2_filtered['wse'].values*100);   
ax_cdf_filter.plot(x, y, color=WSE_color_lake_size_250x250_to_1000x1000_m2, linestyle='--')

x, y = _cdf(df_plot_lake_size_sup_1000x1000_m2_filtered['wse'].values*100);  
ax_cdf_filter.plot(x, y, color=WSE_color_lake_size_sup_1000x1000_m2, linestyle='--')

ax_cdf_filter.set_xlim(-100, 100)
ax_cdf_filter.set_ylim(0, 1)
ax_cdf_filter.set_xlabel("WSE error (cm)")
ax_cdf_filter.set_ylabel('CDF')
ax_cdf_filter.grid(True, which='both', linewidth=0.5)

# Update legend
ax_cdf_filter.legend(fontsize=9)

# Format
yticks = [0.00, 0.16, 0.50, 0.68, 0.84, 1.00]
ax_cdf_filter.set_yticks(yticks)
ax_cdf_filter.set_yticklabels([f"{v:.2f}" for v in yticks])
ax_cdf_filter.grid(True, axis='y', linestyle='--', alpha=0.5)

for ax, label in zip(fig.axes, "acegbdfh"):
    ax.text(
    -0.12, 1.05,
    f"({label})",
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=16,
    )
    ax.tick_params(axis='both', labelsize=12)
    ax.set_xlabel(ax.get_xlabel(), fontsize=14)
    ax.set_ylabel(ax.get_ylabel(), fontsize=14)

ax_cdf_no_filter.set_title("No filtering", fontsize=16, pad = 15, fontweight="medium")
ax_cdf_filter.set_title("Severe filtering", fontsize=16, pad= 15, fontweight="medium")

path_fig = os.path.join(out_dir, "wse_cdf_box_histo_sources.png")
plt.savefig(path_fig, dpi=300)
print(f"Figure saved to : {path_fig}")
plt.close()